# Capstone — Refresh / Content Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YassinKhodir/yassin-flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Author:** Yassin Khodir  
**Question:** Which established content pages should an SEO or content editor review first?  
**Output:** a ranked, human-reviewed queue with reason codes  
**Public paper:** recorded in `submission/paper_url.txt`

## 1. Question

The project supports a limited-capacity review decision: which pages deserve attention first? A high-ranked page is a review candidate, not an automatic refresh order. A false positive wastes editor time; a false negative can leave a meaningful opportunity unnoticed.

In [1]:
from pathlib import Path
import json
import pandas as pd

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "work/outputs/ml09_validation_metrics.json").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the repository or work/notebooks directory.")

repo_root = find_repo_root()
outputs = repo_root / "work/outputs"
figures = repo_root / "work/figures"
print("Decision unit: one pseudonymized client-page record")
print("Human action: inspect, then refresh / rewrite / merge / protect / monitor / no action")

Decision unit: one pseudonymized client-page record
Human action: inspect, then refresh / rewrite / merge / protect / monitor / no action


## 2. Data

- **Release:** FlyRank ML Internship warehouse, daily content performance table.
- **Slice used:** March 2026 partition, 9,841,378 daily rows.
- **Feature window:** March 1–15.
- **Outcome window:** March 16–31.
- **Eligible frame:** 77,400 page records from 38 represented clients.
- **Label:** later average daily impressions more than 20% below the early-window average.
- **Excluded:** all later-window values from features, GA4 fields with uneven coverage, product flags, raw queries, domains, URLs, and IDs as model features.

Requiring rows in both halves of March can favor consistently tracked pages; that population choice is disclosed as a limitation.

In [2]:
data_receipt = pd.Series({
    "march_daily_rows": 9_841_378,
    "eligible_page_rows": 77_400,
    "represented_clients": 38,
    "early_window": "2026-03-01 to 2026-03-15",
    "outcome_window": "2026-03-16 to 2026-03-31",
    "early_impression_floor": 100,
})
data_receipt

march_daily_rows                           9841378
eligible_page_rows                           77400
represented_clients                             38
early_window              2026-03-01 to 2026-03-15
outcome_window            2026-03-16 to 2026-03-31
early_impression_floor                         100
dtype: object

## 3. Methodology

The frozen rule assigns fixed volume and position points. The learned model is logistic regression because the outcome is binary and its probability can rank pages. It uses median imputation, standardization, and seed 42.

The five features are early log impressions, CTR, active days, average position, and position volatility. Entire clients are held out: 28 train clients and 10 test clients. The leakage audit verifies that every feature stops before the later label window; a deliberate label copy produced ROC-AUC 1.000 and was removed.

In [3]:
method_table = pd.DataFrame([
    ["log_early_impressions", "feature", "March 1–15 only"],
    ["early_ctr", "feature", "March 1–15 only"],
    ["early_active_days", "feature", "March 1–15 only"],
    ["early_avg_position", "feature", "March 1–15 only"],
    ["early_position_volatility", "feature", "March 1–15 only"],
    ["is_future_decline", "label", "March 16–31 only"],
    ["client_hash_id", "grouping only", "never a feature"],
    ["content_hash_id", "tie-break/context only", "never a feature"],
], columns=["field", "role", "availability"])
method_table

                       field                    role      availability
0      log_early_impressions                 feature   March 1–15 only
1                  early_ctr                 feature   March 1–15 only
2          early_active_days                 feature   March 1–15 only
3         early_avg_position                 feature   March 1–15 only
4  early_position_volatility                 feature   March 1–15 only
5          is_future_decline                   label  March 16–31 only
6             client_hash_id           grouping only   never a feature
7            content_hash_id  tie-break/context only   never a feature

## 4. Results: model versus baseline

The comparison below uses the same 15,776 held-out rows. The logistic model improved the first 100 slots compared with the frozen rule, but the ROC-AUC of 0.510 shows weak overall separation. That negative result limits the claim.

In [4]:
with open(outputs / "ml08_model_metrics.json") as f:
    model_metrics = json.load(f)

comparison = pd.DataFrame([
    ["Held-out base rate", 0.233, 0.233, 0.233, 0.500],
    ["Frozen rule", 0.300, 0.260, model_metrics["baseline_precision_at_100"], 0.430],
    ["Logistic regression", 0.600, 0.340, model_metrics["model_precision_at_100"], model_metrics["model_roc_auc"]],
], columns=["method", "precision@10", "precision@50", "precision@100", "roc_auc"])
comparison.round(3)

                method  precision@10  precision@50  precision@100  roc_auc
0   Held-out base rate         0.233         0.233          0.233     0.50
1          Frozen rule         0.300         0.260          0.150     0.43
2  Logistic regression         0.600         0.340          0.310     0.51

In [5]:
with open(outputs / "ml09_validation_metrics.json") as f:
    validation_metrics = json.load(f)
validation_comparison = pd.DataFrame([
    ["random pages", validation_metrics["random_split_roc_auc"]],
    ["held-out clients", validation_metrics["grouped_split_roc_auc"]],
    ["deliberate label leak (deleted)", validation_metrics["deliberate_leak_roc_auc"]],
], columns=["validation run", "roc_auc"])
validation_comparison.round(3)

                    validation run  roc_auc
0                     random pages    0.594
1                 held-out clients    0.510
2  deliberate label leak (deleted)    1.000

## 5. Limitations

- One mid-panel month does not represent every season or portfolio.
- Only 10 clients appear in the grouped holdout.
- Eligibility depends on later-window observation, which can create survival bias.
- The label measures later decline, not whether a refresh is useful.
- Five search features omit query intent, content quality, technical state, and business value.
- No controlled intervention was run, so this work cannot claim that refreshing causes recovery.
- Precision@10 is based on ten rows and is too small for a stable headline.

**Safe claim:** on one held-out-client split, the learned score ordered the first 100 candidates better than the fixed rule, while overall discrimination remained weak.

In [6]:
claim_check = {
    "uses_observed_or_measured_language": True,
    "claims_causal_refresh_effect": False,
    "claims_to_predict_google_algorithm": False,
    "requires_human_review": True,
}
assert all([claim_check["uses_observed_or_measured_language"], claim_check["requires_human_review"]])
assert not claim_check["claims_causal_refresh_effect"]
assert not claim_check["claims_to_predict_google_algorithm"]
claim_check

{'uses_observed_or_measured_language': True, 'claims_causal_refresh_effect': False, 'claims_to_predict_google_algorithm': False, 'requires_human_review': True}

## 6. Ranked recommendations

1. Review the high-priority group first; check demand, intent, quality, cannibalization, technical status, business value, and editing cost.
2. Treat `visible_low_ctr` as a title/snippet/intent question, not proof of bad metadata.
3. Treat `high_volume_position_21_50` as a depth/internal-link review cue.
4. Treat `position_unstable` as a reason to wait for a steadier window.
5. Monitor rather than edit when evidence or value is unclear.

Never automate publishing, deletion, merging, redirects, metadata changes, recrawl requests, or expected-lift claims from this score.

In [7]:
with open(outputs / "ml10_playbook_metrics.json") as f:
    playbook_metrics = json.load(f)
pd.Series(playbook_metrics)

queue_rows                15776.000000
held_out_clients             10.000000
high_priority_rows          789.000000
medium_priority_rows       2367.000000
monitor_rows              12620.000000
high_priority_cutoff          0.455889
medium_priority_cutoff        0.420422
grouped_model_roc_auc         0.509776
dtype: float64

## 7. Artifacts the paper embeds

The committed figure below compares the held-out base rate, frozen rule, and logistic model at the first 100 review slots. Aggregate JSON receipts keep the public numbers traceable without committing row-level warehouse exports.

In [8]:
import matplotlib.pyplot as plt

figure_path = figures / "heldout_precision_at_100.png"
assert figure_path.exists()
image = plt.imread(figure_path)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.imshow(image)
ax.axis("off")
plt.show()
print("Paper figure:", figure_path.relative_to(repo_root))
print("Metric receipts:", sorted(p.name for p in outputs.glob("ml0*_*.json")))

Paper figure: work/figures/heldout_precision_at_100.png
Metric receipts: ['ml07_baseline_metrics.json', 'ml08_model_metrics.json', 'ml09_validation_metrics.json']


## 8. Reproducibility

Run the executed notebooks in order: data contract → baseline → model → validation audit → action playbook → capstone. Use Python 3.12 with the repository requirements and a Hugging Face READ token stored as `HF_TOKEN`; never paste it into a notebook. The grouped split uses seed 42. Raw data and row-level CSV queues stay outside git; aggregate metric receipts and the paper figure are committed.

## 9. Acknowledgments and data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai). Thanks to the FlyRank internship team for the pseudonymized warehouse, skeleton notebooks, and public-safety guidance.

## ML-12 — Five-minute demo outline

**0:00–0:40 — Question.** An editor has thousands of pages and limited review time. Which pages deserve attention first?

**0:40–1:25 — Data.** I used the March partition of a pseudonymized search warehouse: early-March features, later-March decline outcome, 77,400 eligible page records.

**1:25–2:20 — Method.** I froze a readable volume-and-position rule, trained a five-feature logistic regression, and held out entire clients. I also tested a deliberate leak and deleted it.

**2:20–3:20 — One chart and honest result.** Precision@100 was 0.310 for the model, 0.150 for the rule, and the held-out base rate was 0.233. But ROC-AUC was only 0.510, so overall separation was weak.

**3:20–4:20 — Recommendation.** Use the score to order a small manual review batch with reason codes. Check intent, quality, technical issues, business value, and editing cost.

**4:20–5:00 — Limit.** Do not automate edits or claim that refreshing causes recovery. The next useful step is testing the queue on more months and tracking editor decisions.

## Shareable cuts

### Short social post

I built a refresh-opportunity ranking study on FlyRank's pseudonymized search warehouse. The main lesson was validation, not model hype: a random page split reached 0.594 ROC-AUC, but holding out entire clients reduced it to 0.510. The model still improved precision@100 over my fixed rule on that split, so I turned it into a small human-review queue with reason codes and clear no-automation limits.

### Employer-facing summary

I built a reproducible content-review ranking workflow using a 9.84M-row monthly partition of pseudonymized search data and a 77,400-row page frame. I compared a transparent baseline with logistic regression under client-held-out validation and ran explicit leakage and error audits. The model improved top-100 prioritization on that split, but overall transfer was weak, so I shipped it as bounded decision support rather than an automated recommendation system.

## Self-check

- [x] Question, unit, user, action, and wrong-call cost are clear
- [x] Data release, table, windows, filters, and exclusions are public-safe
- [x] Features, label, baseline, grouped validation, and leakage audit are documented
- [x] Model, baseline, and base rate use the same held-out rows
- [x] Limitations and the weak overall ROC-AUC are prominent
- [x] Ranked recommendations require human review
- [x] Reproducibility links to committed receipts and a fixed seed
- [x] Abstract and data credit appear on the deployed paper
- [x] ML-12 demo outline, social post, and employer summary are present
- [x] The notebook runs top to bottom without errors